***Leer los primeros 256 bytes del header general***


<p align="center">
    <img src="../../assets/img/headerGeneral.png" alt="Texto alternativo" width="450"/>
</p>

<!-- ![Ejemplo de imagen](../../assets/img/HeaderRecord.png) -->

<!-- Insertar una imag

In [2]:
def read_general_header(edf_path: str) -> bytes:
    with open(edf_path, 'rb') as f:
        header_bytes = f.read(256)
    return header_bytes

data_path = "../dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaauj/s004_2012/01_tcp_ar/aaaaaauj_s004_t000.edf"
header = read_general_header(data_path)
print("Header length:", len(header))
print(header.decode('ascii', errors='replace'))  # reemplaza errores de codificación

FileNotFoundError: [Errno 2] No such file or directory: '../dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaauj/s004_2012/01_tcp_ar/aaaaaauj_s004_t000.edf'

- Ver la informacion que se guarda en el header

| Campo                          | Bytes   | Longitud | Comentario                                  |
| ------------------------------ | ------- | -------- | ------------------------------------------- |
| Version of data format         | 0–7     | 8        | Versión del formato EDF |
| Local patient identification   | 8–87    | 80       | Información del paciente                      |
| Local recording identification | 88–167  | 80       | Información sobre el estudio o grabacion                |
| Start date of recording        | 168–175 | 8        | Fecha de inicio de la grabación en formato dd.mm.yy                         |
| Start time of recording        | 176–183 | 8        | Hora de inicio en formato hh.mm.ss                         |
| Number of bytes in header      | 184–191 | 8        | Número de bytes totales del header                          |
| Reserved                       | 192–235 | 44       | Uso libre, algunos ponen etiquetas          |
| Number of data records         | 236–243 | 8        | Número de bloques de datos ("registros")                        |
| Duration of a data record (s)  | 244–251 | 8        |  Duración de cada registro de datos, en segundos        |
| Number of signals (channels)   | 252–255 | 4        | Número total de señales o canales                            |


In [4]:
version = header[0:8].decode().strip()
local_patient_id = header[8:88].decode().strip()
local_recording_id = header[88:168].decode().strip()
start_date = header[168:176].decode().strip()
start_time = header[176:184].decode().strip()
header_length = header[184:192].decode().strip()
reserved = header[192:236].decode().strip()
number_of_data_records = header[236:244].decode().strip()
duration_of_data_record = header[244:252].decode().strip()
n_signals = int(header[252:256].decode().strip())

print(f"Versión EDF: {version}")
print(f"ID del paciente: {local_patient_id}")
print(f"ID de la grabación: {local_recording_id}")
print(f"Fecha inicio: {start_date}")
print(f"Hora inicio: {start_time}")
print(f"Longitud del encabezado: {header_length}")
print(f"Reservado: {reserved}")
print(f"Número de registros de datos: {number_of_data_records}")
print(f"Duración de cada registro de datos: {duration_of_data_record}")
print(f"Número de canales: {n_signals}")

Versión EDF: 0
ID del paciente: aaaaaauj F 01-JAN-0000 aaaaaauj Age:77
ID de la grabación: Startdate 01-JAN-2012 aaaaaauj_s004 XXX X
Fecha inicio: 01.01.12
Hora inicio: 00.00.00
Longitud del encabezado: 7936
Reservado: 
Número de registros de datos: 1245
Duración de cada registro de datos: 1.000000
Número de canales: 30


In [ ]:
256 + 256*30 # 7936

7936

In [ ]:
from typing import Any, Dict, List
import pyedflib
import os

def extract_patient_id_from_path(path: str) -> str:
    parts: str = os.path.normpath(path).split(os.sep)
    return parts[-4] if len(parts) >= 4 else "unknown"

def extract_edf_metadata(edf_paths: List[str]) -> List[Dict[str, Any]]:
    metadata: List[Dict[str, Any]] = []
    for path in edf_paths:
        try:
            f = pyedflib.EdfReader(path)

            # in read byte, similar to number_of_data_records = header[236:244].decode().strip()
            duration_sec = f.file_duration 

            # in read byte, similar to signals_in_file = header[252:255].decode().strip()
            n_channels = f.signals_in_file
            
            patient_id = extract_patient_id_from_path(path)
            filename = os.path.basename(path)

            metadata.append({
                "filename": filename,
                "filepath": path,
                "patient_id": patient_id,
                "duration_sec": duration_sec,
                "n_channels": n_channels
            })
            f._close()
            del f
        except Exception as e:
            print(f"❌ Error procesando {path}: {e}")
    return metadata

extract_edf_metadata()